# 📦 Notebook 5: Batching and Aggregation

Instead of writing every event immediately, batch them together or aggregate counters to reduce write load.

## Learning Objectives

By the end of this notebook, you'll understand:
- Write batching at different layers
- Counter aggregation patterns
- Hierarchical aggregation for fan-in
- Trade-offs of batching

In [1]:
import psycopg2
import redis
import time
import random
from datetime import datetime
from collections import defaultdict
from typing import Dict, List

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "writes_demo",
    "user": "demo",
    "password": "demo",
}

def get_connection():
    c = psycopg2.connect(**DB_CONFIG)
    c.autocommit = True
    return c

conn = get_connection()
r = redis.Redis(host="localhost", port=6379, decode_responses=True)
r.ping()

print("✅ Connected to PostgreSQL and Redis!")
print("📊 Open Adminer:      http://localhost:8080")
print("📊 Open RedisInsight: http://localhost:5540")


✅ Connected to PostgreSQL and Redis!
📊 Open Adminer:      http://localhost:8080
📊 Open RedisInsight: http://localhost:5540


## 📦 Batching Layers

In [2]:
print("📦 Where to Batch Writes")
print("=" * 60)
print("""
                    Batching Layers
─────────────────────────────────────────────────────────────

┌─────────────────────────────────────────────────────────┐
│                     APPLICATION                          │
│   • Buffer writes in memory                             │
│   • Flush every N items or T seconds                    │
│   • Risk: Data loss on crash                            │
└───────────────────────┬─────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────┐
│                  INTERMEDIATE LAYER                      │
│   • Redis/Kafka as write buffer                         │
│   • Durable, can survive app crashes                    │
│   • Background workers batch to DB                      │
└───────────────────────┬─────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────┐
│                      DATABASE                            │
│   • COPY command (PostgreSQL)                           │
│   • Bulk insert APIs                                    │
│   • Write-ahead log batching                            │
└─────────────────────────────────────────────────────────┘
""")

📦 Where to Batch Writes

                    Batching Layers
─────────────────────────────────────────────────────────────

┌─────────────────────────────────────────────────────────┐
│                     APPLICATION                          │
│   • Buffer writes in memory                             │
│   • Flush every N items or T seconds                    │
│   • Risk: Data loss on crash                            │
└───────────────────────┬─────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────┐
│                  INTERMEDIATE LAYER                      │
│   • Redis/Kafka as write buffer                         │
│   • Durable, can survive app crashes                    │
│   • Background workers batch to DB                      │
└───────────────────────┬─────────────────────────────────┘
                        ▼
┌─────────────────────────────────────────────────────────┐
│                      DATABASE           

In [3]:
class ApplicationBatcher:
    def __init__(self, batch_size: int = 100, flush_interval: float = 1.0):
        self.batch_size = batch_size
        self.flush_interval = flush_interval
        self.buffer: List[dict] = []
        self.last_flush = time.time()
        self.total_writes = 0
        self.total_flushes = 0
    
    def add(self, item: dict):
        self.buffer.append(item)
        self.total_writes += 1
        
        if len(self.buffer) >= self.batch_size:
            self.flush()
        elif time.time() - self.last_flush > self.flush_interval:
            self.flush()
    
    def flush(self):
        if not self.buffer:
            return
        
        self.total_flushes += 1
        batch_size = len(self.buffer)
        self.buffer = []
        self.last_flush = time.time()
        
        return batch_size
    
    def get_stats(self) -> dict:
        return {
            "total_writes": self.total_writes,
            "total_flushes": self.total_flushes,
            "db_write_reduction": f"{(1 - self.total_flushes/max(1, self.total_writes))*100:.1f}%"
        }

print("🔬 Application-Level Batching")
print("=" * 60)

batcher = ApplicationBatcher(batch_size=50)

print("\n📝 Writing 500 items with batch size 50...")
for i in range(500):
    batcher.add({"id": i, "value": f"data_{i}"})

batcher.flush()

stats = batcher.get_stats()
print(f"\n📊 Results:")
print(f"   Total writes: {stats['total_writes']}")
print(f"   Total flushes: {stats['total_flushes']}")
print(f"   DB write reduction: {stats['db_write_reduction']}")

print("\n✅ 500 writes → 10 database operations!")

🔬 Application-Level Batching

📝 Writing 500 items with batch size 50...

📊 Results:
   Total writes: 500
   Total flushes: 10
   DB write reduction: 98.0%

✅ 500 writes → 10 database operations!


## 👍 Counter Aggregation

In [4]:
print("👍 The Like Counter Problem")
print("=" * 60)
print("""
PROBLEM: Viral post gets 10,000 likes per second
─────────────────────────────────────────────────────────────

NAIVE: Each like = 1 database write
    UPDATE posts SET likes = likes + 1 WHERE id = 123
    
    10,000 writes/second → Database explodes! 💥

─────────────────────────────────────────────────────────────

SOLUTION: Aggregate in Redis, periodically sync to DB

    ┌─────────┐      ┌─────────┐      ┌──────────┐
    │  Likes  │ ───> │  Redis  │ ───> │ Database │
    │ 10K/sec │      │ INCR    │      │ 1/minute │
    └─────────┘      │ counter │      └──────────┘
                     └─────────┘
    
    • Redis handles 100K+ INCR/sec
    • Database sees 1 write/minute per post
    • 600,000x write reduction!
""")

👍 The Like Counter Problem

PROBLEM: Viral post gets 10,000 likes per second
─────────────────────────────────────────────────────────────

NAIVE: Each like = 1 database write
    UPDATE posts SET likes = likes + 1 WHERE id = 123

    10,000 writes/second → Database explodes! 💥

─────────────────────────────────────────────────────────────

SOLUTION: Aggregate in Redis, periodically sync to DB

    ┌─────────┐      ┌─────────┐      ┌──────────┐
    │  Likes  │ ───> │  Redis  │ ───> │ Database │
    │ 10K/sec │      │ INCR    │      │ 1/minute │
    └─────────┘      │ counter │      └──────────┘
                     └─────────┘

    • Redis handles 100K+ INCR/sec
    • Database sees 1 write/minute per post
    • 600,000x write reduction!



In [5]:
class LikeAggregator:
    def __init__(self, conn):
        self.conn = conn
        self.prefix = "like_count"
        self.likes_received = 0
        self.syncs_performed = 0
    
    def add_like(self, post_id: int):
        key = f"{self.prefix}:{post_id}"
        r.incr(key)
        self.likes_received += 1
    
    def sync_to_db(self):
        cursor = self.conn.cursor()
        keys = r.keys(f"{self.prefix}:*")
        
        for key in keys:
            count = int(r.getdel(key) or 0)
            if count > 0:
                post_id = int(key.split(":")[1])
                cursor.execute(
                    "UPDATE post_metrics SET like_count = like_count + %s WHERE post_id = %s",
                    (count, post_id)
                )
                self.syncs_performed += 1
        
        cursor.close()

cursor = conn.cursor()
cursor.execute("DELETE FROM post_metrics")
for i in range(10):
    cursor.execute("INSERT INTO post_metrics (post_id, like_count) VALUES (%s, 0)", (i,))
cursor.close()

print("🔬 Like Aggregation Demo")
print("=" * 60)

aggregator = LikeAggregator(conn)

print("\n👍 Simulating 10,000 likes across 10 posts...")
import random
for _ in range(10000):
    post_id = random.randint(0, 9)
    aggregator.add_like(post_id)

print(f"   Likes received: {aggregator.likes_received}")
print(f"   Keys in Redis: {len(r.keys(f'{aggregator.prefix}:*'))}")

print("\n🔄 Syncing to database...")
aggregator.sync_to_db()

print(f"   Database writes: {aggregator.syncs_performed}")
print(f"   Write reduction: {(1 - aggregator.syncs_performed/aggregator.likes_received)*100:.2f}%")

cursor = conn.cursor()
cursor.execute("SELECT post_id, like_count FROM post_metrics ORDER BY like_count DESC LIMIT 5")
print("\n📊 Top 5 posts by likes:")
for row in cursor.fetchall():
    print(f"   Post {row[0]}: {row[1]} likes")
cursor.close()

print("\n✅ 10,000 likes → 10 database writes!")

🔬 Like Aggregation Demo

👍 Simulating 10,000 likes across 10 posts...


   Likes received: 10000
   Keys in Redis: 10

🔄 Syncing to database...
   Database writes: 10
   Write reduction: 99.90%

📊 Top 5 posts by likes:
   Post 8: 1053 likes
   Post 9: 1050 likes
   Post 2: 1027 likes
   Post 4: 1018 likes
   Post 5: 1007 likes

✅ 10,000 likes → 10 database writes!


## 🌳 Hierarchical Aggregation

In [6]:
print("🌳 Hierarchical Aggregation")
print("=" * 60)
print("""
PROBLEM: 1000 servers all writing to 1 database
─────────────────────────────────────────────────────────────

    Server 1 ──┐
    Server 2 ──┤
    Server 3 ──┼───> [Single Database] 💥 Bottleneck!
       ...     │
    Server N ──┘

─────────────────────────────────────────────────────────────

SOLUTION: Hierarchical aggregation (fan-in)

    Level 0 (1000 servers):
    ┌───┐ ┌───┐ ┌───┐     ┌───┐
    │ S1│ │ S2│ │ S3│ ... │S1K│  Buffer locally
    └─┬─┘ └─┬─┘ └─┬─┘     └─┬─┘
      │     │     │         │
      └──┬──┘     └────┬────┘
    
    Level 1 (100 aggregators):
       ┌───────┐      ┌───────┐
       │ Agg 1 │      │ Agg N │   Batch 10 servers each
       └───┬───┘      └───┬───┘
           │              │
           └──────┬───────┘
    
    Level 2 (10 aggregators):
             ┌─────────┐
             │ Final   │
             │ Agg     │   Batch 10 aggregators each
             └────┬────┘
                  │
                  ▼
            [Database]  ← Only 10 writers!

1000 servers → 10 database connections
""")

🌳 Hierarchical Aggregation

PROBLEM: 1000 servers all writing to 1 database
─────────────────────────────────────────────────────────────

    Server 1 ──┐
    Server 2 ──┤
    Server 3 ──┼───> [Single Database] 💥 Bottleneck!
       ...     │
    Server N ──┘

─────────────────────────────────────────────────────────────

SOLUTION: Hierarchical aggregation (fan-in)

    Level 0 (1000 servers):
    ┌───┐ ┌───┐ ┌───┐     ┌───┐
    │ S1│ │ S2│ │ S3│ ... │S1K│  Buffer locally
    └─┬─┘ └─┬─┘ └─┬─┘     └─┬─┘
      │     │     │         │
      └──┬──┘     └────┬────┘

    Level 1 (100 aggregators):
       ┌───────┐      ┌───────┐
       │ Agg 1 │      │ Agg N │   Batch 10 servers each
       └───┬───┘      └───┬───┘
           │              │
           └──────┬───────┘

    Level 2 (10 aggregators):
             ┌─────────┐
             │ Final   │
             │ Agg     │   Batch 10 aggregators each
             └────┬────┘
                  │
                  ▼
            [Database]  

In [7]:
class HierarchicalAggregator:
    def __init__(self, level: int, batch_threshold: int = 10):
        self.level = level
        self.batch_threshold = batch_threshold
        self.buffer: Dict[str, int] = defaultdict(int)
        self.upstream: 'HierarchicalAggregator' = None
        self.flushes = 0
        
    def set_upstream(self, upstream: 'HierarchicalAggregator'):
        self.upstream = upstream
    
    def write(self, key: str, value: int = 1):
        self.buffer[key] += value
        
        if sum(self.buffer.values()) >= self.batch_threshold:
            self.flush()
    
    def flush(self):
        if not self.buffer:
            return
            
        self.flushes += 1
        
        if self.upstream:
            for key, value in self.buffer.items():
                self.upstream.write(key, value)
        
        self.buffer.clear()

print("🔬 Hierarchical Aggregation Demo")
print("=" * 60)

db_writer = HierarchicalAggregator(level=2, batch_threshold=100)

mid_aggregators = [HierarchicalAggregator(level=1, batch_threshold=50) for _ in range(10)]
for agg in mid_aggregators:
    agg.set_upstream(db_writer)

servers = [HierarchicalAggregator(level=0, batch_threshold=10) for _ in range(100)]
for i, server in enumerate(servers):
    server.set_upstream(mid_aggregators[i // 10])

print("\n📝 100 servers each writing 100 events...")
for server in servers:
    for _ in range(100):
        key = f"metric_{random.randint(0, 9)}"
        server.write(key)

for server in servers:
    server.flush()
for agg in mid_aggregators:
    agg.flush()
db_writer.flush()

server_flushes = sum(s.flushes for s in servers)
mid_flushes = sum(a.flushes for a in mid_aggregators)
db_flushes = db_writer.flushes

print(f"\n📊 Results:")
print(f"   Total events: 10,000")
print(f"   Level 0 (servers) flushes: {server_flushes}")
print(f"   Level 1 (mid-tier) flushes: {mid_flushes}")
print(f"   Level 2 (database) flushes: {db_flushes}")
print(f"   Write reduction: {(1 - db_flushes/10000)*100:.2f}%")

print("\n✅ 10,000 events → ~100 database writes!")

🔬 Hierarchical Aggregation Demo

📝 100 servers each writing 100 events...

📊 Results:
   Total events: 10,000
   Level 0 (servers) flushes: 1000
   Level 1 (mid-tier) flushes: 200
   Level 2 (database) flushes: 100
   Write reduction: 99.00%

✅ 10,000 events → ~100 database writes!


## 🧪 Quick Quiz

1. **What's the trade-off of counter aggregation?**

2. **When would you use hierarchical aggregation?**

3. **What happens if an aggregator crashes before syncing?**

In [8]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Counter aggregation trade-off:")
print("   - Counts are eventually consistent")
print("   - May show slightly stale values")
print("   - Sync failure = lost increments")
print()
print("2. Use hierarchical aggregation when:")
print("   - Many sources writing to one destination")
print("   - Fan-in pattern (1000s → 1)")
print("   - Example: metrics from 1000 servers")
print()
print("3. Aggregator crash:")
print("   - In-memory buffer is lost")
print("   - Solution: Write-ahead log (WAL)")
print("   - Or use Redis as durable buffer")

📝 Quiz Answers

1. Counter aggregation trade-off:
   - Counts are eventually consistent
   - May show slightly stale values
   - Sync failure = lost increments

2. Use hierarchical aggregation when:
   - Many sources writing to one destination
   - Fan-in pattern (1000s → 1)
   - Example: metrics from 1000 servers

3. Aggregator crash:
   - In-memory buffer is lost
   - Solution: Write-ahead log (WAL)
   - Or use Redis as durable buffer


## 📚 Summary

### Key Takeaways

1. **Batch at multiple layers** - App, middleware, database
2. **Counter aggregation** - Buffer in Redis, sync periodically
3. **Hierarchical aggregation** - For fan-in patterns
4. **Trade-offs** - Eventual consistency, crash recovery
5. **Massive reduction** - 10,000x fewer database writes possible

### Next Up

In **Notebook 6**, we'll learn about hot keys:
- Detecting hot keys
- Key splitting strategies
- Dynamic resharding